# Policy Lever Impact Analysis

This notebook evaluates how key policy levers affect permitting outcomes.

Levers included:
- Staffing levels (planning, public works, fire)
- Permit volume (`num_permits`)

For each scenario, the notebook runs multiple Monte Carlo repetitions and summarizes impacts on:
- Mean total processing time
- Median total processing time
- Completion count
- Mean staff utilization (planning, public works, fire)


In [ ]:
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from run_simulation import run_multiple_simulations, plot_staff_utilization_series

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


In [ ]:
# --- Baseline + policy scenario grid ---
BASE_PARAMS = {
    "name": "baseline",
    "sequential": "standard",
    "ai_review": "none",
    "pct_pre_approved": 0.018,
    "pct_custom": 0.91,
    "pct_self_cert": 0.081,
    "pct_like_for_like": 0.803,
    "pre_application_distribution": "baseline",
    "review_duration_families": None,
    "review_duration_multipliers": None,
    "planning_staff_count": 20,
    "planning_caseload_per_staff": 7,
    "public_works_staff_count": 30,
    "public_works_caseload_per_staff": 7,
    "fire_staff_count": 10,
    "fire_caseload_per_staff": 7,
}

N_RUNS = 20
BASE_SEED = 42
SIMULATION_DURATION = None

NUM_PERMITS_OPTIONS = [2000, 6500]

# Scenario dimensions (not policy levers)
STAFFING_SCENARIOS = {
    "low": {"planning_staff_count": 2, "public_works_staff_count": 3, "fire_staff_count": 1},
    "medium": {"planning_staff_count": 8, "public_works_staff_count": 12, "fire_staff_count": 4},
    "high": {"planning_staff_count": 20, "public_works_staff_count": 30, "fire_staff_count": 10},
}
PRE_APPLICATION_DISTRIBUTIONS = ["lognormal_10", "lognormal_60", "lognormal_180"]

# Policy levers
SEQUENTIAL_OPTIONS = ["sequential", "standard", "parallel"]
AI_REVIEW_OPTIONS = ["none", "initial_check", "full_review"]


def build_scenarios():
    scenarios = []
    for (
        num_permits,
        (staffing_name, staffing),
        pre_app_dist,
        sequential_mode,
        ai_mode,
    ) in itertools.product(
        NUM_PERMITS_OPTIONS,
        STAFFING_SCENARIOS.items(),
        PRE_APPLICATION_DISTRIBUTIONS,
        SEQUENTIAL_OPTIONS,
        AI_REVIEW_OPTIONS,
    ):
        scenario = dict(BASE_PARAMS)
        scenario["name"] = (
            f"permits={num_permits}|staffing={staffing_name}|"
            f"preapp={pre_app_dist}|seq={sequential_mode}|ai={ai_mode}"
        )
        scenario["planning_staff_count"] = staffing["planning_staff_count"]
        scenario["public_works_staff_count"] = staffing["public_works_staff_count"]
        scenario["fire_staff_count"] = staffing["fire_staff_count"]
        scenario["pre_application_distribution"] = pre_app_dist
        scenario["sequential"] = sequential_mode
        scenario["ai_review"] = ai_mode
        scenarios.append((num_permits, staffing_name, pre_app_dist, sequential_mode, ai_mode, scenario))
    return scenarios


SCENARIOS = build_scenarios()
print(f"Total scenarios: {len(SCENARIOS)}")
print("Staffing scenarios:", list(STAFFING_SCENARIOS.keys()))
print("Pre-application distributions:", PRE_APPLICATION_DISTRIBUTIONS)
print("Policy levers:")
print("  sequential:", SEQUENTIAL_OPTIONS)
print("  ai_review:", AI_REVIEW_OPTIONS)


In [ ]:
# --- Run experiments ---
rows = []
util_by_scenario = {}

for idx, (num_permits, staffing_name, pre_app_dist, sequential_mode, ai_mode, scenario) in enumerate(SCENARIOS, start=1):
    results, avg_util_by_scenario = run_multiple_simulations(
        n_runs=N_RUNS,
        num_permits=num_permits,
        simulation_duration=SIMULATION_DURATION,
        base_seed=BASE_SEED,
        scenario_params_list=[scenario],
        collect_permits=True,
        collect_average_staff_utilization=True,
        utilization_step=0.1,
    )

    util = avg_util_by_scenario[scenario["name"]]
    util_by_scenario[scenario["name"]] = util

    app_to_ready_mean_by_run = []
    app_to_ready_median_by_run = []
    completed_by_run = []

    for r in results:
        permits = r.get("permits", [])
        durations = [
            (p.ready_for_construction - p.planning_request)
            for p in permits
            if p.ready_for_construction is not None and p.planning_request is not None
        ]
        if durations:
            app_to_ready_mean_by_run.append(float(np.mean(durations)))
            app_to_ready_median_by_run.append(float(np.median(durations)))
        else:
            app_to_ready_mean_by_run.append(np.nan)
            app_to_ready_median_by_run.append(np.nan)
        completed_by_run.append(len(durations))

    rows.append(
        {
            "scenario": scenario["name"],
            "num_permits": num_permits,
            "staffing_scenario": staffing_name,
            "pre_application_distribution": pre_app_dist,
            "sequential": sequential_mode,
            "ai_review": ai_mode,
            "planning_staff_count": scenario["planning_staff_count"],
            "public_works_staff_count": scenario["public_works_staff_count"],
            "fire_staff_count": scenario["fire_staff_count"],
            "application_to_ready_mean_days": float(np.nanmean(app_to_ready_mean_by_run)),
            "application_to_ready_mean_days_std": float(np.nanstd(app_to_ready_mean_by_run, ddof=1) if len(app_to_ready_mean_by_run) > 1 else 0.0),
            "application_to_ready_median_days": float(np.nanmean(app_to_ready_median_by_run)),
            "completed_mean": float(np.mean(completed_by_run)),
            "planning_util_mean_pct": float(100.0 * np.mean(util["planning"])),
            "planning_util_peak_pct": float(100.0 * np.max(util["planning"])),
            "public_works_util_mean_pct": float(100.0 * np.mean(util["public_works"])),
            "public_works_util_peak_pct": float(100.0 * np.max(util["public_works"])),
            "fire_util_mean_pct": float(100.0 * np.mean(util["fire"])),
            "fire_util_peak_pct": float(100.0 * np.max(util["fire"])),
        }
    )

    if idx % 10 == 0 or idx == len(SCENARIOS):
        print(f"Completed {idx}/{len(SCENARIOS)} scenarios")

impact_df = pd.DataFrame(rows)
impact_df.head()


In [ ]:
# --- Policy lever impacts (parallel vs sequential, AI initial vs full) ---
def effect_span(df, lever, metric):
    grouped = df.groupby(lever, as_index=False)[metric].mean().sort_values(lever)
    return grouped, grouped[metric].max() - grouped[metric].min()

metrics = [
    "application_to_ready_mean_days",
    "application_to_ready_median_days",
    "planning_util_mean_pct",
    "public_works_util_mean_pct",
    "fire_util_mean_pct",
]

policy_levers = ["sequential", "ai_review"]

for metric in metrics:
    print("\n" + "=" * 90)
    print(f"Metric: {metric}")
    print("=" * 90)
    spans = []
    for lever in policy_levers:
        grouped, span = effect_span(impact_df, lever, metric)
        spans.append((lever, span))
        print(f"\nPolicy lever: {lever} (span={span:.2f})")
        display(grouped)

    print("\nPolicy lever rank by effect span:")
    for lever, span in sorted(spans, key=lambda x: x[1], reverse=True):
        print(f"  {lever:24s} {span:10.2f}")

print("\n" + "=" * 90)
print("Policy levers within each staffing / permit / pre-app scenario")
print("=" * 90)
strata_cols = ["num_permits", "staffing_scenario", "pre_application_distribution"]
for keys, group in impact_df.groupby(strata_cols):
    print(f"\nStratum: permits={keys[0]}, staffing={keys[1]}, pre_app={keys[2]}")
    pivot = group.pivot_table(
        index="sequential",
        columns="ai_review",
        values="application_to_ready_mean_days",
        aggfunc="mean",
    )
    display(pivot)


In [ ]:
# --- Plot average utilization for selected scenarios ---
# Pick a few scenarios to compare; adjust list as needed.
selected = impact_df.sort_values("application_to_ready_mean_days").head(3)["scenario"].tolist()
selected += impact_df.sort_values("application_to_ready_mean_days", ascending=False).head(3)["scenario"].tolist()
selected = list(dict.fromkeys(selected))

for scenario_name in selected:
    util = util_by_scenario[scenario_name]
    plot_staff_utilization_series(
        util,
        as_percent=True,
        title=f"Mean staff utilization across {util['n_runs']} runs\n{scenario_name}",
        xlim=(0, min(1000, util["max_day"])),
        ylim=(0, 100),
    )


In [ ]:
# --- Save results for reuse ---
from pathlib import Path

results_dir = Path("results")
results_dir.mkdir(parents=True, exist_ok=True)

output_path = results_dir / "policy_lever_impact_results.csv"
impact_df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")


### Figures: comparisons by cohort size and staffing

Loads `results/policy_lever_impact_results.csv` and builds **six** bar charts (permits × staffing).

Bars are **grouped by pre-application distribution** (lognormal 10 / 60 / 180). Within each group, five bars show policies in this order: standard/no AI, standard/initial review, standard/full review, sequential/no AI, parallel/no AI.

Figures are written under `results/`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RESULTS_CSV = Path("results") / "policy_lever_impact_results.csv"
df_plot = pd.read_csv(RESULTS_CSV)

POLICY_ROWS = [
    ("standard", "none", "Standard / no AI"),
    ("standard", "initial_check", "Standard / initial review"),
    ("standard", "full_review", "Standard / full review"),
    ("sequential", "none", "Sequential / no AI"),
    ("parallel", "none", "Parallel / no AI"),
]
PRE_APP_LEVELS = ["lognormal_10", "lognormal_60", "lognormal_180"]
PRE_APP_LABELS = {
    "lognormal_10": "Pre-app:\nlognormal 10",
    "lognormal_60": "Pre-app:\nlognormal 60",
    "lognormal_180": "Pre-app:\nlognormal 180",
}

policy_colors = [
    "#d62728",  # Standard / no AI (red)
    "#ff7f0e",  # Standard / initial review (orange)
    "#f2c300",  # Standard / full review (yellow)
    "#2ca02c",  # Sequential / no AI (green)
    "#1f77b4",  # Parallel / no AI (blue)
]

results_dir = Path("results")
results_dir.mkdir(parents=True, exist_ok=True)

for num_permits in sorted(df_plot["num_permits"].unique()):
    for staffing in ["low", "medium", "high"]:
        subset = df_plot[
            (df_plot["num_permits"] == num_permits)
            & (df_plot["staffing_scenario"] == staffing)
            & (df_plot["pre_application_distribution"].isin(PRE_APP_LEVELS))
        ]

        n_policies = len(POLICY_ROWS)
        n_groups = len(PRE_APP_LEVELS)
        x = np.arange(n_groups)
        total_width = 0.85
        bar_width = total_width / n_policies

        fig, ax = plt.subplots(figsize=(11, 5.5))
        for i, (seq, ai, policy_label) in enumerate(POLICY_ROWS):
            heights = []
            for pre in PRE_APP_LEVELS:
                row = subset[
                    (subset["sequential"] == seq)
                    & (subset["ai_review"] == ai)
                    & (subset["pre_application_distribution"] == pre)
                ]
                if len(row) != 1:
                    raise ValueError(
                        f"Expected 1 row for permits={num_permits}, staffing={staffing}, "
                        f"pre={pre}, sequential={seq}, ai={ai}; got {len(row)}"
                    )
                heights.append(float(row["application_to_ready_mean_days"].iloc[0]))
            offset = bar_width * (i - (n_policies - 1) / 2)
            ax.bar(x + offset, heights, bar_width, label=policy_label, color=policy_colors[i], edgecolor="white", linewidth=0.6)

        ax.set_xticks(x)
        ax.set_xticklabels([PRE_APP_LABELS[p] for p in PRE_APP_LEVELS], fontsize=10)
        ax.set_ylabel("Mean days (planning request → ready for construction)")
        ax.set_title(f"Permits = {num_permits:,} · Staffing = {staffing}")
        ax.legend(title="Policy scenario", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
        ax.grid(axis="y", alpha=0.3)
        plt.tight_layout()
        out_path = results_dir / f"app_to_ready_policy_comparison_permits{num_permits}_staffing_{staffing}.png"
        fig.savefig(out_path, dpi=150, bbox_inches="tight")
        plt.show()
        print(out_path)
